<a href="https://colab.research.google.com/github/ypg1um-arch/SAU_ML_TASKS/blob/main/PMMHA_Ablation_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# base_models.py code[as the repo clone is temporary]
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import Linear
from torch_geometric.nn.inits import glorot
from torch_geometric.nn.conv import MessagePassing
from torch import Tensor


class MLPEncoder(nn.Module):
    def __init__(self, in_dims, hid_dims, dropout_rate: float = 0.0, negative_slope: float = 0.2):
        super().__init__()

        self.encoder_layers = nn.Sequential(
            Linear(in_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros'),
            nn.LeakyReLU(negative_slope),
            nn.Dropout(p=dropout_rate),
            Linear(hid_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros')
        )

    def forward(self, x):
        x = self.encoder_layers(x)
        return x
# fuse() function_______________________________________________________________
def fuse(x_proj, mask, mode="original", att_lin=None, shared_weights=None):
    """
    Standalone fusion function for MAGNET Ablation Study.
    """
    batch_size, num_heads, num_modalities, head_dims = x_proj.size()

    if mode == "original":
        assert att_lin is not None, "att_lin parameter is required for 'original' mode"
        att_scores = torch.matmul(x_proj, att_lin.transpose(-1, -2)).squeeze(-1)
        att_scores = att_scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        att_weights = torch.softmax(att_scores, dim=-1)

    elif mode == "equal":
        counts = mask.sum(dim=1, keepdim=True).clamp(min=1)
        equal_weights = mask / counts
        att_weights = equal_weights.unsqueeze(1).expand(-1, num_heads, -1)

    elif mode == "shared":
        assert shared_weights is not None, "shared_weights parameter is required for 'shared' mode"
        shared_scores = shared_weights.view(1, 1, num_modalities).expand(batch_size, num_heads, -1)
        shared_scores = shared_scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        att_weights = torch.softmax(shared_scores, dim=-1)

    else:
        raise ValueError(f"Unknown mode: {mode}. Choose 'equal', 'original', or 'shared'.")

    att_weights = att_weights * mask.unsqueeze(1)
    fused_embeddings = torch.sum(att_weights.unsqueeze(-1) * x_proj, dim=2)

    return fused_embeddings, att_weights
#_______________________________________________________________________________
class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, hid_dims, num_heads, num_modalities=3):
        super().__init__()
        self.num_heads = num_heads
        self.head_dims = hid_dims // num_heads
        assert (
            self.head_dims * num_heads == hid_dims
        ), "hid_dims must be divisible by num_heads"

        self.lin_proj = Linear(hid_dims, hid_dims, bias=False, weight_initializer='glorot')
        self.att_lin = nn.Parameter(torch.empty(num_heads, 1, self.head_dims))
        self.out_proj = Linear(hid_dims, hid_dims, bias=False, weight_initializer='glorot')

        # Register the shared learnable weights for Mode C here
        self.shared_weights = nn.Parameter(torch.zeros(num_modalities))

        self.reset_parameters()

    def reset_parameters(self):
        self.lin_proj.reset_parameters()
        self.out_proj.reset_parameters()
        glorot(self.att_lin)
        nn.init.normal_(self.shared_weights, mean=0.0, std=0.1)

    def forward(self, x, mask, mode="original"):
        """
        x: [batch_size, num_modalities, hid_dims] - Patient embeddings for all modalities
        mask: [batch_size, num_modalities] - Mask indicating available modalities
        """
        batch_size, num_modalities, hid_dims = x.size()

        # 1. Linear projection
        x_proj = self.lin_proj(x).view(batch_size, num_modalities, self.num_heads, self.head_dims)
        x_proj = x_proj.permute(0, 2, 1, 3)

        # ─── FORCE ACTIVE MODE HERE ───────────────────────────────────────
        current_mode = "shared"  # Ensure this is set to "shared"
        # ───────────────────────────────────────────────────────────────────

        # ─── FORCE UNIQUE SHARED WEIGHTS ──────────────────────────────────
        if current_mode == "shared":
            # This forces a fixed global bias: heavily favoring DNA over mRNA and miRNA
            with torch.no_grad():
                self.shared_weights.copy_(torch.tensor([2.5, -1.0, -1.5], device=x.device))
        # ───────────────────────────────────────────────────────────────────

        if not getattr(self, '_has_printed_mode', False):
            print(f"\n[MAGNET EXECUTION] >>> Current Fusion Mode Active: {current_mode.upper()} <<<\n")
            if current_mode == "shared":
                probs = torch.softmax(self.shared_weights, dim=0).detach().cpu().numpy()
                print(f"[FORCED STATIC BIAS] DNA: {probs[0]:.4f}, mRNA: {probs[1]:.4f}, miRNA: {probs[2]:.4f}\n")
            self._has_printed_mode = True

        # 2. Call standalone fuse function
        fused_embeddings, att_weights = fuse(
            x_proj=x_proj,
            mask=mask,
            mode=current_mode,
            att_lin=self.att_lin,
            shared_weights=self.shared_weights
        )

        # 3. Concatenate heads and project output
        fused_embeddings = fused_embeddings.view(batch_size, -1)
        output = self.out_proj(fused_embeddings)

        return output, att_weights


class EdgeSAGEConv(MessagePassing):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        aggr = "mean",
        bias: bool = True,
        edge_dim: int = None,
        **kwargs,
    ):
        super().__init__(aggr, **kwargs)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.edge_dim = edge_dim
        in_channels = (in_channels, in_channels)

        if self.edge_dim is not None:
            self.lin_msg = Linear(in_channels[0] + self.edge_dim, in_channels[0], weight_initializer='glorot', bias_initializer='zeros', bias=True)

        self.lin = Linear(in_channels[0], in_channels[0], weight_initializer='glorot', bias_initializer='zeros', bias=True)
        self.lin_l = Linear(in_channels[0], out_channels, weight_initializer='glorot', bias_initializer='zeros', bias=bias)
        self.lin_r = Linear(in_channels[1], out_channels, weight_initializer='glorot', bias_initializer='zeros', bias=False)

        self.act_msg = nn.ReLU()

        self.reset_parameters()

    def reset_parameters(self):
        super().reset_parameters()
        self.lin.reset_parameters()
        self.lin_l.reset_parameters()
        self.lin_r.reset_parameters()
        if self.edge_dim is not None:
            self.lin_msg.reset_parameters()

    def forward(self, x, edge_index, edge_attr = None):
        if isinstance(x, Tensor):
            x = (x, x)

        x = (self.lin(x[0]).relu(), x[1])

        out = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        out = self.lin_l(out)
        x_r = x[1]
        out = out + self.lin_r(x_r)

        return out

    def message(self, x_j, edge_attr = None):
        if edge_attr is not None and self.edge_dim is not None:
            if edge_attr.dim() == 1:
                edge_attr = edge_attr.unsqueeze(-1)
            msg = torch.cat([x_j, edge_attr], dim=-1)
            return self.act_msg(self.lin_msg(msg))
        return x_j


class GNNDecoder(nn.Module):
    def __init__(self, hid_dims, out_dims, num_layers, dropout_rate: float = 0.0, negative_slope: float = 0.2):
        super().__init__()

        self.dropout_rate = dropout_rate
        self.negative_slope = negative_slope
        self.conv_layers = nn.ModuleList()

        for _ in range(num_layers):
            self.conv_layers.append(EdgeSAGEConv(hid_dims, hid_dims, edge_dim=1))

        self.decoder_layers = nn.Sequential(
            Linear(hid_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros'),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate)
        )
        self.final_layer = Linear(hid_dims, out_dims, weight_initializer='glorot', bias_initializer='zeros')


    def forward(self, x, edge_index, edge_attr=None, return_embedding=False):
        for conv in self.conv_layers:
            x = conv(x, edge_index, edge_attr=edge_attr)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout_rate, training=self.training)

        x = self.decoder_layers(x)
        embeddings = x

        logits = self.final_layer(x)

        if return_embedding:
            return logits, embeddings

        return logits

In [ ]:
# base_models.py code[as the repo clone is temporary]
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import Linear
from torch_geometric.nn.inits import glorot
from torch_geometric.nn.conv import MessagePassing
from torch import Tensor


class MLPEncoder(nn.Module):
    def __init__(self, in_dims, hid_dims, dropout_rate: float = 0.0, negative_slope: float = 0.2):
        super().__init__()

        self.encoder_layers = nn.Sequential(
            Linear(in_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros'),
            nn.LeakyReLU(negative_slope),
            nn.Dropout(p=dropout_rate),
            Linear(hid_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros')
        )

    def forward(self, x):
        x = self.encoder_layers(x)
        return x
# fuse() function_______________________________________________________________
def fuse(x_proj, mask, mode="original", att_lin=None, shared_weights=None):
    """
    Standalone fusion function for MAGNET Ablation Study.
    """
    batch_size, num_heads, num_modalities, head_dims = x_proj.size()

    if mode == "original":
        assert att_lin is not None, "att_lin parameter is required for 'original' mode"
        att_scores = torch.matmul(x_proj, att_lin.transpose(-1, -2)).squeeze(-1)
        att_scores = att_scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        att_weights = torch.softmax(att_scores, dim=-1)

    elif mode == "equal":
        counts = mask.sum(dim=1, keepdim=True).clamp(min=1)
        equal_weights = mask / counts
        att_weights = equal_weights.unsqueeze(1).expand(-1, num_heads, -1)

    elif mode == "shared":
        assert shared_weights is not None, "shared_weights parameter is required for 'shared' mode"
        shared_scores = shared_weights.view(1, 1, num_modalities).expand(batch_size, num_heads, -1)
        shared_scores = shared_scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        att_weights = torch.softmax(shared_scores, dim=-1)

    else:
        raise ValueError(f"Unknown mode: {mode}. Choose 'equal', 'original', or 'shared'.")

    att_weights = att_weights * mask.unsqueeze(1)
    fused_embeddings = torch.sum(att_weights.unsqueeze(-1) * x_proj, dim=2)

    return fused_embeddings, att_weights
#_______________________________________________________________________________
class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, hid_dims, num_heads, num_modalities=3):
        super().__init__()
        self.num_heads = num_heads
        self.head_dims = hid_dims // num_heads
        assert (
            self.head_dims * num_heads == hid_dims
        ), "hid_dims must be divisible by num_heads"

        self.lin_proj = Linear(hid_dims, hid_dims, bias=False, weight_initializer='glorot')
        self.att_lin = nn.Parameter(torch.empty(num_heads, 1, self.head_dims))
        self.out_proj = Linear(hid_dims, hid_dims, bias=False, weight_initializer='glorot')

        # Register the shared learnable weights for Mode C here
        self.shared_weights = nn.Parameter(torch.zeros(num_modalities))

        self.reset_parameters()

    def reset_parameters(self):
        self.lin_proj.reset_parameters()
        self.out_proj.reset_parameters()
        glorot(self.att_lin)
        nn.init.normal_(self.shared_weights, mean=0.0, std=0.1)

    def forward(self, x, mask, mode="original"):
        """
        x: [batch_size, num_modalities, hid_dims] - Patient embeddings for all modalities
        mask: [batch_size, num_modalities] - Mask indicating available modalities
        """
        batch_size, num_modalities, hid_dims = x.size()

        # 1. Linear projection
        x_proj = self.lin_proj(x).view(batch_size, num_modalities, self.num_heads, self.head_dims)
        x_proj = x_proj.permute(0, 2, 1, 3)

        # ─── FORCE ACTIVE MODE HERE ───────────────────────────────────────
        current_mode = "shared"  # Ensure this is set to "shared"
        # ───────────────────────────────────────────────────────────────────

        # ─── FORCE UNIQUE SHARED WEIGHTS ──────────────────────────────────
        if current_mode == "shared":
            # This forces a fixed global bias: heavily favoring DNA over mRNA and miRNA
            with torch.no_grad():
                self.shared_weights.copy_(torch.tensor([2.5, -1.0, -1.5], device=x.device))
        # ───────────────────────────────────────────────────────────────────

        if not getattr(self, '_has_printed_mode', False):
            print(f"\n[MAGNET EXECUTION] >>> Current Fusion Mode Active: {current_mode.upper()} <<<\n")
            if current_mode == "shared":
                probs = torch.softmax(self.shared_weights, dim=0).detach().cpu().numpy()
                print(f"[FORCED STATIC BIAS] DNA: {probs[0]:.4f}, mRNA: {probs[1]:.4f}, miRNA: {probs[2]:.4f}\n")
            self._has_printed_mode = True

        # 2. Call standalone fuse function
        fused_embeddings, att_weights = fuse(
            x_proj=x_proj,
            mask=mask,
            mode=current_mode,
            att_lin=self.att_lin,
            shared_weights=self.shared_weights
        )

        # 3. Concatenate heads and project output
        fused_embeddings = fused_embeddings.view(batch_size, -1)
        output = self.out_proj(fused_embeddings)

        return output, att_weights


class EdgeSAGEConv(MessagePassing):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        aggr = "mean",
        bias: bool = True,
        edge_dim: int = None,
        **kwargs,
    ):
        super().__init__(aggr, **kwargs)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.edge_dim = edge_dim
        in_channels = (in_channels, in_channels)

        if self.edge_dim is not None:
            self.lin_msg = Linear(in_channels[0] + self.edge_dim, in_channels[0], weight_initializer='glorot', bias_initializer='zeros', bias=True)

        self.lin = Linear(in_channels[0], in_channels[0], weight_initializer='glorot', bias_initializer='zeros', bias=True)
        self.lin_l = Linear(in_channels[0], out_channels, weight_initializer='glorot', bias_initializer='zeros', bias=bias)
        self.lin_r = Linear(in_channels[1], out_channels, weight_initializer='glorot', bias_initializer='zeros', bias=False)

        self.act_msg = nn.ReLU()

        self.reset_parameters()

    def reset_parameters(self):
        super().reset_parameters()
        self.lin.reset_parameters()
        self.lin_l.reset_parameters()
        self.lin_r.reset_parameters()
        if self.edge_dim is not None:
            self.lin_msg.reset_parameters()

    def forward(self, x, edge_index, edge_attr = None):
        if isinstance(x, Tensor):
            x = (x, x)

        x = (self.lin(x[0]).relu(), x[1])

        out = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        out = self.lin_l(out)
        x_r = x[1]
        out = out + self.lin_r(x_r)

        return out

    def message(self, x_j, edge_attr = None):
        if edge_attr is not None and self.edge_dim is not None:
            if edge_attr.dim() == 1:
                edge_attr = edge_attr.unsqueeze(-1)
            msg = torch.cat([x_j, edge_attr], dim=-1)
            return self.act_msg(self.lin_msg(msg))
        return x_j


class GNNDecoder(nn.Module):
    def __init__(self, hid_dims, out_dims, num_layers, dropout_rate: float = 0.0, negative_slope: float = 0.2):
        super().__init__()

        self.dropout_rate = dropout_rate
        self.negative_slope = negative_slope
        self.conv_layers = nn.ModuleList()

        for _ in range(num_layers):
            self.conv_layers.append(EdgeSAGEConv(hid_dims, hid_dims, edge_dim=1))

        self.decoder_layers = nn.Sequential(
            Linear(hid_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros'),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate)
        )
        self.final_layer = Linear(hid_dims, out_dims, weight_initializer='glorot', bias_initializer='zeros')


    def forward(self, x, edge_index, edge_attr=None, return_embedding=False):
        for conv in self.conv_layers:
            x = conv(x, edge_index, edge_attr=edge_attr)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout_rate, training=self.training)

        x = self.decoder_layers(x)
        embeddings = x

        logits = self.final_layer(x)

        if return_embedding:
            return logits, embeddings

        return logits

In [1]:
# 1. Clone the repository
!git clone https://github.com/SinaTabakhi/MAGNET.git
%cd MAGNET

# 2. Detect Colab's current PyTorch version dynamically
import torch
torch_version = torch.__version__.split('+')[0]
cuda_version = torch.version.cuda.replace('.', '')
print(f"Colab is using Torch {torch_version} with CUDA {cuda_version}")

# 3. Install PyG binaries that match Colab's environment exactly
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{torch_version}+cu{cuda_version}.html
!pip install torch-geometric==2.4.0

# 4. Install the remaining requirements
!pip install lightning==2.1.3 pandas matplotlib umap-learn yacs comet_ml "ray[tune]"

Cloning into 'MAGNET'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 202 (delta 8), reused 0 (delta 0), pack-reused 167 (from 1)
Receiving objects: 100% (202/202), 53.31 MiB | 27.16 MiB/s, done.
Resolving deltas: 100% (49/49), done.
Updating files: 100% (151/151), done.
/content/MAGNET
Colab is using Torch 2.11.0 with CUDA 128
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 115.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 133.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 70.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [7]:
%cd /content/MAGNET

/content/MAGNET


In [6]:
!python main_inference.py --cfg configs/MAGNET_OV.yaml

Epoch 169: 100% 1/1 [00:00<00:00,  6.85it/s, v_num=0, train_kl_loss=0.0755, train_cls_loss=0.0117, train_total_loss=0.0192, train_acc=0.997, train_auroc=1.000, train_auprc=1.000, train_f1=0.996, train_mcc=0.993] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation:   0% 0/1 [00:00<?, ?it/s]        
Validation DataLoader 0:   0% 0/1 [00:00<?, ?it/s]
Validation DataLoader 0: 100% 1/1 [00:00<00:00, 2314.74it/s]
Epoch 170: 100% 1/1 [00:00<00:00,  6.17it/s, v_num=0, train_kl_loss=0.0762, train_cls_loss=0.00769, train_total_loss=0.0153, train_acc=1.000, train_auroc=1.000, train_auprc=1.000, train_f1=1.000, train_mcc=1.000]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation:   0% 0/1 [00:00<?, ?it/s]        
Validation DataLoader 0:   0% 0/1 [00:00<?, ?it/s]
Validation DataLoader 0: 100% 1/1 [00:00<00:00, 2179.99it/s]
Epoch 171: 100% 1/1 [00:00<00:00,  6.63it/s, v_num=0, train_kl_loss=0.0805, train_cls_loss=0.0149, train_total_loss=0.0229, train_acc=1.000, train_auroc=1.000, train_

In [ ]:
# base_models.py code[as the repo clone is temporary](use for original + equal modes ONLY)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import Linear
from torch_geometric.nn.inits import glorot
from torch_geometric.nn.conv import MessagePassing
from torch import Tensor


class MLPEncoder(nn.Module):
    def __init__(self, in_dims, hid_dims, dropout_rate: float = 0.0, negative_slope: float = 0.2):
        super().__init__()

        self.encoder_layers = nn.Sequential(
            Linear(in_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros'),
            nn.LeakyReLU(negative_slope),
            nn.Dropout(p=dropout_rate),
            Linear(hid_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros')
        )

    def forward(self, x):
        x = self.encoder_layers(x)
        return x
# fuse() function_______________________________________________________________
def fuse(x_proj, mask, mode="original", att_lin=None, shared_weights=None):
    """
    Standalone fusion function for MAGNET Ablation Study.
    """
    batch_size, num_heads, num_modalities, head_dims = x_proj.size()

    if mode == "original":
        assert att_lin is not None, "att_lin parameter is required for 'original' mode"
        att_scores = torch.matmul(x_proj, att_lin.transpose(-1, -2)).squeeze(-1)
        att_scores = att_scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        att_weights = torch.softmax(att_scores, dim=-1)

    elif mode == "equal":
        counts = mask.sum(dim=1, keepdim=True).clamp(min=1)
        equal_weights = mask / counts
        att_weights = equal_weights.unsqueeze(1).expand(-1, num_heads, -1)

    elif mode == "shared":
        assert shared_weights is not None, "shared_weights parameter is required for 'shared' mode"
        shared_scores = shared_weights.view(1, 1, num_modalities).expand(batch_size, num_heads, -1)
        shared_scores = shared_scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        att_weights = torch.softmax(shared_scores, dim=-1)

    else:
        raise ValueError(f"Unknown mode: {mode}. Choose 'equal', 'original', or 'shared'.")

    att_weights = att_weights * mask.unsqueeze(1)
    fused_embeddings = torch.sum(att_weights.unsqueeze(-1) * x_proj, dim=2)

    return fused_embeddings, att_weights
#_______________________________________________________________________________
class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, hid_dims, num_heads, num_modalities=3):
        super().__init__()
        self.num_heads = num_heads
        self.head_dims = hid_dims // num_heads
        assert (
            self.head_dims * num_heads == hid_dims
        ), "hid_dims must be divisible by num_heads"

        self.lin_proj = Linear(hid_dims, hid_dims, bias=False, weight_initializer='glorot')
        self.att_lin = nn.Parameter(torch.empty(num_heads, 1, self.head_dims))
        self.out_proj = Linear(hid_dims, hid_dims, bias=False, weight_initializer='glorot')

        # Register the shared learnable weights for Mode C here
        self.shared_weights = nn.Parameter(torch.zeros(num_modalities))

        self.reset_parameters()

    def reset_parameters(self):
        self.lin_proj.reset_parameters()
        self.out_proj.reset_parameters()
        glorot(self.att_lin)
        nn.init.normal_(self.shared_weights, mean=0.0, std=0.1)

    def forward(self, x, mask, mode="original"):
        """
        x: [batch_size, num_modalities, hid_dims] - Patient embeddings for all modalities
        mask: [batch_size, num_modalities] - Mask indicating available modalities
        """
        batch_size, num_modalities, hid_dims = x.size()

        # 1. Linear projection
        x_proj = self.lin_proj(x).view(batch_size, num_modalities, self.num_heads, self.head_dims)
        x_proj = x_proj.permute(0, 2, 1, 3)

        # ─── FORCE ACTIVE MODE HERE ───────────────────────────────────────
        current_mode = "shared"  # Ensure this is set to "shared"
        # ───────────────────────────────────────────────────────────────────

        # ─── FORCE UNIQUE SHARED WEIGHTS ──────────────────────────────────
        if current_mode == "shared":
            # This forces a fixed global bias: heavily favoring DNA over mRNA and miRNA
            with torch.no_grad():
                self.shared_weights.copy_(torch.tensor([2.5, -1.0, -1.5], device=x.device))
        # ───────────────────────────────────────────────────────────────────

        if not getattr(self, '_has_printed_mode', False):
            print(f"\n[MAGNET EXECUTION] >>> Current Fusion Mode Active: {current_mode.upper()} <<<\n")
            if current_mode == "shared":
                probs = torch.softmax(self.shared_weights, dim=0).detach().cpu().numpy()
                print(f"[FORCED STATIC BIAS] DNA: {probs[0]:.4f}, mRNA: {probs[1]:.4f}, miRNA: {probs[2]:.4f}\n")
            self._has_printed_mode = True

        # 2. Call standalone fuse function
        fused_embeddings, att_weights = fuse(
            x_proj=x_proj,
            mask=mask,
            mode=current_mode,
            att_lin=self.att_lin,
            shared_weights=self.shared_weights
        )

        # 3. Concatenate heads and project output
        fused_embeddings = fused_embeddings.view(batch_size, -1)
        output = self.out_proj(fused_embeddings)

        return output, att_weights


class EdgeSAGEConv(MessagePassing):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        aggr = "mean",
        bias: bool = True,
        edge_dim: int = None,
        **kwargs,
    ):
        super().__init__(aggr, **kwargs)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.edge_dim = edge_dim
        in_channels = (in_channels, in_channels)

        if self.edge_dim is not None:
            self.lin_msg = Linear(in_channels[0] + self.edge_dim, in_channels[0], weight_initializer='glorot', bias_initializer='zeros', bias=True)

        self.lin = Linear(in_channels[0], in_channels[0], weight_initializer='glorot', bias_initializer='zeros', bias=True)
        self.lin_l = Linear(in_channels[0], out_channels, weight_initializer='glorot', bias_initializer='zeros', bias=bias)
        self.lin_r = Linear(in_channels[1], out_channels, weight_initializer='glorot', bias_initializer='zeros', bias=False)

        self.act_msg = nn.ReLU()

        self.reset_parameters()

    def reset_parameters(self):
        super().reset_parameters()
        self.lin.reset_parameters()
        self.lin_l.reset_parameters()
        self.lin_r.reset_parameters()
        if self.edge_dim is not None:
            self.lin_msg.reset_parameters()

    def forward(self, x, edge_index, edge_attr = None):
        if isinstance(x, Tensor):
            x = (x, x)

        x = (self.lin(x[0]).relu(), x[1])

        out = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        out = self.lin_l(out)
        x_r = x[1]
        out = out + self.lin_r(x_r)

        return out

    def message(self, x_j, edge_attr = None):
        if edge_attr is not None and self.edge_dim is not None:
            if edge_attr.dim() == 1:
                edge_attr = edge_attr.unsqueeze(-1)
            msg = torch.cat([x_j, edge_attr], dim=-1)
            return self.act_msg(self.lin_msg(msg))
        return x_j


class GNNDecoder(nn.Module):
    def __init__(self, hid_dims, out_dims, num_layers, dropout_rate: float = 0.0, negative_slope: float = 0.2):
        super().__init__()

        self.dropout_rate = dropout_rate
        self.negative_slope = negative_slope
        self.conv_layers = nn.ModuleList()

        for _ in range(num_layers):
            self.conv_layers.append(EdgeSAGEConv(hid_dims, hid_dims, edge_dim=1))

        self.decoder_layers = nn.Sequential(
            Linear(hid_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros'),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate)
        )
        self.final_layer = Linear(hid_dims, out_dims, weight_initializer='glorot', bias_initializer='zeros')


    def forward(self, x, edge_index, edge_attr=None, return_embedding=False):
        for conv in self.conv_layers:
            x = conv(x, edge_index, edge_attr=edge_attr)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout_rate, training=self.training)

        x = self.decoder_layers(x)
        embeddings = x

        logits = self.final_layer(x)

        if return_embedding:
            return logits, embeddings

        return logits

In [ ]:
#new version of base_models.py(use for shared mode ONLY)
# base_models.py code[as the repo clone is temporary]
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import Linear
from torch_geometric.nn.inits import glorot
from torch_geometric.nn.conv import MessagePassing
from torch import Tensor


class MLPEncoder(nn.Module):
    def __init__(self, in_dims, hid_dims, dropout_rate: float = 0.0, negative_slope: float = 0.2):
        super().__init__()

        self.encoder_layers = nn.Sequential(
            Linear(in_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros'),
            nn.LeakyReLU(negative_slope),
            nn.Dropout(p=dropout_rate),
            Linear(hid_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros')
        )

    def forward(self, x):
        x = self.encoder_layers(x)
        return x
# fuse() function_______________________________________________________________
def fuse(x_proj, mask, mode="original", att_lin=None, shared_weights=None):
    """
    Standalone fusion function for MAGNET Ablation Study.
    """
    batch_size, num_heads, num_modalities, head_dims = x_proj.size()

    if mode == "original":
        assert att_lin is not None, "att_lin parameter is required for 'original' mode"
        att_scores = torch.matmul(x_proj, att_lin.transpose(-1, -2)).squeeze(-1)
        att_scores = att_scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        att_weights = torch.softmax(att_scores, dim=-1)

    elif mode == "equal":
        counts = mask.sum(dim=1, keepdim=True).clamp(min=1)
        equal_weights = mask / counts
        att_weights = equal_weights.unsqueeze(1).expand(-1, num_heads, -1)

    elif mode == "shared":
        assert shared_weights is not None, "shared_weights parameter is required for 'shared' mode"
        shared_scores = shared_weights.view(1, 1, num_modalities).expand(batch_size, num_heads, -1)
        shared_scores = shared_scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        att_weights = torch.softmax(shared_scores, dim=-1)

    else:
        raise ValueError(f"Unknown mode: {mode}. Choose 'equal', 'original', or 'shared'.")

    att_weights = att_weights * mask.unsqueeze(1)
    fused_embeddings = torch.sum(att_weights.unsqueeze(-1) * x_proj, dim=2)

    return fused_embeddings, att_weights
#_______________________________________________________________________________
class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, hid_dims, num_heads, num_modalities=3):
        super().__init__()
        self.num_heads = num_heads
        self.head_dims = hid_dims // num_heads
        assert (
            self.head_dims * num_heads == hid_dims
        ), "hid_dims must be divisible by num_heads"

        self.lin_proj = Linear(hid_dims, hid_dims, bias=False, weight_initializer='glorot')
        self.att_lin = nn.Parameter(torch.empty(num_heads, 1, self.head_dims))
        self.out_proj = Linear(hid_dims, hid_dims, bias=False, weight_initializer='glorot')

        # Register the shared learnable weights for Mode C here
        self.shared_weights = nn.Parameter(torch.zeros(num_modalities))

        self.reset_parameters()

    def reset_parameters(self):
        self.lin_proj.reset_parameters()
        self.out_proj.reset_parameters()
        glorot(self.att_lin)
        nn.init.normal_(self.shared_weights, mean=0.0, std=0.1)

    def forward(self, x, mask, mode="original"):
        """
        x: [batch_size, num_modalities, hid_dims] - Patient embeddings for all modalities
        mask: [batch_size, num_modalities] - Mask indicating available modalities
        """
        batch_size, num_modalities, hid_dims = x.size()

        # 1. Linear projection
        x_proj = self.lin_proj(x).view(batch_size, num_modalities, self.num_heads, self.head_dims)
        x_proj = x_proj.permute(0, 2, 1, 3)

        # ─── FORCE ACTIVE MODE HERE ───────────────────────────────────────
        current_mode = "shared"  # Ensure this is set to "shared"
        # ───────────────────────────────────────────────────────────────────

        if not getattr(self, '_has_printed_mode', False):
            print(f"\n[MAGNET EXECUTION] >>> Current Fusion Mode Active: {current_mode.upper()} <<<\n")
            if current_mode == "shared":
                probs = torch.softmax(self.shared_weights, dim=0).detach().cpu().numpy()
                print(f"[FORCED STATIC BIAS] DNA: {probs[0]:.4f}, mRNA: {probs[1]:.4f}, miRNA: {probs[2]:.4f}\n")
            self._has_printed_mode = True

        # 2. Call standalone fuse function
        fused_embeddings, att_weights = fuse(
            x_proj=x_proj,
            mask=mask,
            mode=current_mode,
            att_lin=self.att_lin,
            shared_weights=self.shared_weights
        )

        # 3. Concatenate heads and project output
        fused_embeddings = fused_embeddings.view(batch_size, -1)
        output = self.out_proj(fused_embeddings)

        return output, att_weights


class EdgeSAGEConv(MessagePassing):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        aggr = "mean",
        bias: bool = True,
        edge_dim: int = None,
        **kwargs,
    ):
        super().__init__(aggr, **kwargs)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.edge_dim = edge_dim
        in_channels = (in_channels, in_channels)

        if self.edge_dim is not None:
            self.lin_msg = Linear(in_channels[0] + self.edge_dim, in_channels[0], weight_initializer='glorot', bias_initializer='zeros', bias=True)

        self.lin = Linear(in_channels[0], in_channels[0], weight_initializer='glorot', bias_initializer='zeros', bias=True)
        self.lin_l = Linear(in_channels[0], out_channels, weight_initializer='glorot', bias_initializer='zeros', bias=bias)
        self.lin_r = Linear(in_channels[1], out_channels, weight_initializer='glorot', bias_initializer='zeros', bias=False)

        self.act_msg = nn.ReLU()

        self.reset_parameters()

    def reset_parameters(self):
        super().reset_parameters()
        self.lin.reset_parameters()
        self.lin_l.reset_parameters()
        self.lin_r.reset_parameters()
        if self.edge_dim is not None:
            self.lin_msg.reset_parameters()

    def forward(self, x, edge_index, edge_attr = None):
        if isinstance(x, Tensor):
            x = (x, x)

        x = (self.lin(x[0]).relu(), x[1])

        out = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        out = self.lin_l(out)
        x_r = x[1]
        out = out + self.lin_r(x_r)

        return out

    def message(self, x_j, edge_attr = None):
        if edge_attr is not None and self.edge_dim is not None:
            if edge_attr.dim() == 1:
                edge_attr = edge_attr.unsqueeze(-1)
            msg = torch.cat([x_j, edge_attr], dim=-1)
            return self.act_msg(self.lin_msg(msg))
        return x_j


class GNNDecoder(nn.Module):
    def __init__(self, hid_dims, out_dims, num_layers, dropout_rate: float = 0.0, negative_slope: float = 0.2):
        super().__init__()

        self.dropout_rate = dropout_rate
        self.negative_slope = negative_slope
        self.conv_layers = nn.ModuleList()

        for _ in range(num_layers):
            self.conv_layers.append(EdgeSAGEConv(hid_dims, hid_dims, edge_dim=1))

        self.decoder_layers = nn.Sequential(
            Linear(hid_dims, hid_dims, weight_initializer='glorot', bias_initializer='zeros'),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate)
        )
        self.final_layer = Linear(hid_dims, out_dims, weight_initializer='glorot', bias_initializer='zeros')


    def forward(self, x, edge_index, edge_attr=None, return_embedding=False):
        for conv in self.conv_layers:
            x = conv(x, edge_index, edge_attr=edge_attr)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout_rate, training=self.training)

        x = self.decoder_layers(x)
        embeddings = x

        logits = self.final_layer(x)

        if return_embedding:
            return logits, embeddings

        return logits

In [ ]:
#modified trainer.py
from typing import List

import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
import lightning as L

from model.base_models import MLPEncoder, MultiHeadAttentionLayer, GNNDecoder
from dataloader.dataloader import MultiomicsDataset
from utils import align_modalities, evaluate_classification_performance


class MAGNETTrainer(L.LightningModule):
    def __init__(
        self,
        dataset: MultiomicsDataset,
        unimodal_encoders: List[MLPEncoder],
        attention_layer: MultiHeadAttentionLayer,
        gnn_decoder: GNNDecoder,
        loss_fn: CrossEntropyLoss,
        lr: float,
        wd: float
    ):
        super().__init__()
        self.save_hyperparameters("lr", "wd")
        self.dataset = dataset
        self.unimodal_encoders = nn.ModuleList(unimodal_encoders)
        self.attention_layer = attention_layer
        self.gnn_decoder = gnn_decoder
        self.loss_fn = loss_fn
        self.lr = lr
        self.wd = wd

        self.train_embeddings_to_plot = {}
        self.train_labels_to_plot = {}
        self.test_embeddings_to_plot = {}
        self.test_labels_to_plot = {}

    def get_embeddings(self):
        return self.train_embeddings_to_plot, self.train_labels_to_plot, self.test_embeddings_to_plot, self.test_labels_to_plot

    def configure_optimizers(self):
        optimizer = Adam(self.parameters(), lr=self.lr, weight_decay=self.wd)
        scheduler = StepLR(optimizer, step_size=20, gamma=0.8)

        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'interval': 'epoch',
                'frequency': 1
            }
        }

    def similarity_kl_loss(self, similarities, embeddings, mask=None, alpha=1.0, eps=1e-12):
        device = embeddings.device
        n = embeddings.size(0)

        # Compute squared distances (Euclidean distance)
        sum_sq = torch.sum(embeddings ** 2, dim=1)
        dist_sq = sum_sq.unsqueeze(1) + sum_sq.unsqueeze(0) - 2 * torch.matmul(embeddings, embeddings.T)

        # Student-t kernel similarity
        q_matrix = torch.pow(1 + dist_sq / alpha, -(alpha + 1) / 2)

        # Remove self-similarity
        q_matrix = q_matrix * (1 - torch.eye(n, device=device))

        # Apply mask if provided
        if mask is not None:
            q_matrix = q_matrix.masked_fill(mask == 0, 0.0)
            similarities = similarities.masked_fill(mask == 0, 0.0)

        # Normalize to probability distributions (P and Q)
        q_distribution = torch.clamp(q_matrix / (q_matrix.sum() + eps), min=eps)
        p_distribution = torch.clamp(similarities / (similarities.sum() + eps), min=eps)

        # KL divergence: KL(P || Q)
        kl_div = torch.sum(p_distribution * torch.log(p_distribution / q_distribution))

        return kl_div

    def forward(self, x, mode, return_embedding=False):
        data, data_indices, modality_mask, labels = x
        encoded_modalities = []

        for modality_idx in range(self.dataset.num_omics):
            model_output = self.unimodal_encoders[modality_idx](data[modality_idx])
            encoded_modalities.append(model_output)

        aligned_modalities = align_modalities(encoded_modalities, data_indices, labels)
        modality_embeddings = torch.stack(aligned_modalities, dim=1)
        fused_embeddings, _ = self.attention_layer(modality_embeddings, modality_mask)

        graph_data, similarities, edge_mask = self.dataset.build_graph_data(fused_embeddings, mode)

        if return_embedding:
            output, gnn_embeddings = self.gnn_decoder(graph_data.x, graph_data.edge_index, graph_data.edge_attr, return_embedding=True)
            return aligned_modalities, fused_embeddings, gnn_embeddings, graph_data

        output = self.gnn_decoder(graph_data.x, graph_data.edge_index, graph_data.edge_attr)

        return output, graph_data, similarities, fused_embeddings, edge_mask

    def training_step(self, batch, batch_idx):
        output, graph_data, similarities, fused_embeddings, edge_mask = self(batch, mode="train")
        train_kl_loss = self.similarity_kl_loss(similarities, fused_embeddings, edge_mask)

        mask = graph_data.train_mask
        y_true = graph_data.y.squeeze()[mask]
        y_logit = output[mask]
        train_cls_loss = self.loss_fn(y_logit, y_true)

        total_loss = train_cls_loss + 0.1 * train_kl_loss

        probs = torch.softmax(y_logit, dim=-1)
        metrics = evaluate_classification_performance(y_true, probs, self.dataset.num_classes)

        self.log('train_kl_loss', train_kl_loss.detach().cpu(), prog_bar=True)
        self.log('train_cls_loss', train_cls_loss.detach().cpu(), prog_bar=True)
        self.log('train_total_loss', total_loss.detach().cpu(), prog_bar=True)

        for name, value in metrics.items():
            self.log(f'train_{name}', value, prog_bar=True)

        return total_loss

    def on_before_optimizer_step(self, optimizer):
        # This hook runs automatically right after loss.backward()

        # We check if gradients exist, and print them occasionally so it doesn't flood your console
        if torch.rand(1).item() < 0.05:  # Prints roughly 5% of the time
            grad = self.attention_layer.shared_weights.grad
            if grad is not None:
                print(f"\n[GRADIENT CHECK] Shared weights gradient: {grad.detach().cpu().numpy()}")
            else:
                print("\n[GRADIENT CHECK] Shared weights gradient is NONE (Warning: Not attached to loss!)")

    def validation_step(self, batch, batch_idx):
        if self.dataset.tune_hyperparameters:
            output, graph_data, similarities, fused_embeddings, edge_mask = self(batch, mode="val")
            val_kl_loss = self.similarity_kl_loss(similarities, fused_embeddings, edge_mask)

            mask = graph_data.val_mask
            y_true = graph_data.y.squeeze()[mask]
            y_logit = output[mask]
            val_cls_loss = self.loss_fn(y_logit, y_true)

            total_loss = val_cls_loss + 0.1 * val_kl_loss

            probs = torch.softmax(y_logit, dim=-1)
            metrics = evaluate_classification_performance(y_true, probs, self.dataset.num_classes)

            self.log('val_kl_loss', val_kl_loss.detach().cpu(), prog_bar=True)
            self.log('val_cls_loss', val_cls_loss.detach().cpu(), prog_bar=True)
            self.log('val_total_loss', total_loss.detach().cpu(), prog_bar=True)

            for name, value in metrics.items():
                self.log(f'val_{name}', value, prog_bar=True)

    def test_step(self, batch, batch_idx):
        output, graph_data, similarities, fused_embeddings, edge_mask = self(batch, mode="test")
        test_kl_loss = self.similarity_kl_loss(similarities, fused_embeddings, edge_mask)

        mask = graph_data.test_mask
        y_true = graph_data.y.squeeze()[mask]
        y_logit = output[mask]
        test_cls_loss = self.loss_fn(y_logit, y_true)

        total_loss = test_cls_loss + 0.1 * test_kl_loss

        probs = torch.softmax(y_logit, dim=-1)
        metrics = evaluate_classification_performance(y_true, probs, self.dataset.num_classes)

        self.log('test_kl_loss', test_kl_loss.detach().cpu(), prog_bar=True)
        self.log('test_cls_loss', test_cls_loss.detach().cpu(), prog_bar=True)
        self.log('test_total_loss', total_loss.detach().cpu(), prog_bar=True)

        for name, value in metrics.items():
            self.log(f'test_{name}', value, prog_bar=True)

        # Paired patient mask
        mask_paired = graph_data.test_mask_paired
        y_true_paired = graph_data.y.squeeze()[mask_paired]
        y_logit_paired = output[mask_paired]
        probs_paired = torch.softmax(y_logit_paired, dim=-1)

        metrics_paired = evaluate_classification_performance(y_true_paired, probs_paired, self.dataset.num_classes)
        for name, value in metrics_paired.items():
            self.log(f'test_paired_{name}', value, prog_bar=False)

        # Unpaired patient mask
        mask_unpaired = graph_data.test_mask_unpaired
        y_true_unpaired = graph_data.y.squeeze()[mask_unpaired]
        if len(y_true_unpaired) != 0:
            y_logit_unpaired = output[mask_unpaired]
            probs_unpaired = torch.softmax(y_logit_unpaired, dim=-1)

            metrics_unpaired = evaluate_classification_performance(y_true_unpaired, probs_unpaired, self.dataset.num_classes)
            for name, value in metrics_unpaired.items():
                if value is not None:
                    self.log(f'test_unpaired_{name}', value, prog_bar=False)
                else:
                    self.log(f'test_unpaired_{name}', float('nan'), prog_bar=False)

    def on_test_end(self) -> None:
        batch = self.dataset[0]
        batch = self.transfer_batch_to_device(batch=batch, device=self.device, dataloader_idx=0)
        aligned_modalities, fused_embeddings, gnn_embeddings, graph_data = self(batch, mode="train", return_embedding=True)
        self.train_embeddings_to_plot, self.train_labels_to_plot = self._process_embeddings(aligned_modalities,
                                                                                            fused_embeddings,
                                                                                            gnn_embeddings,
                                                                                            graph_data,
                                                                                            graph_data.train_mask,
                                                                                            mode="train")

        aligned_modalities, fused_embeddings, gnn_embeddings, graph_data = self(batch, mode="test", return_embedding=True)
        self.test_embeddings_to_plot, self.test_labels_to_plot =  self._process_embeddings(aligned_modalities,
                                                                                           fused_embeddings,
                                                                                           gnn_embeddings,
                                                                                           graph_data,
                                                                                           graph_data.test_mask,
                                                                                           mode="test")

    def _process_embeddings(self, aligned_modalities, fused_embeddings, gnn_embeddings, graph_data, mask, mode):
        embeddings_to_plot = {}
        labels_to_plot = {}

        full_labels = graph_data.y[mask].detach().cpu().numpy()

        for idx, data in enumerate(aligned_modalities):
            mask_data = data[mask].detach().cpu().numpy()
            non_zero_rows = ~torch.all(data[mask] == 0, dim=1).detach().cpu().numpy()
            filter_data = mask_data[non_zero_rows]
            filter_labels = full_labels[non_zero_rows]

            if mode == "train":
                embeddings_to_plot[f"Input ({self.dataset.modalities[idx]})"] = self.dataset.get_data(
                    idx).detach().cpu().numpy()[:len(filter_data)]
            else:
                embeddings_to_plot[f"Input ({self.dataset.modalities[idx]})"] = self.dataset.get_data(
                    idx).detach().cpu().numpy()[-len(filter_data):]
            labels_to_plot[f"Input ({self.dataset.modalities[idx]})"] = filter_labels

            embeddings_to_plot[f"Embedded ({self.dataset.modalities[idx]})"] = filter_data
            labels_to_plot[f"Embedded ({self.dataset.modalities[idx]})"] = filter_labels

        fused_mask = fused_embeddings[mask].detach().cpu().numpy()
        gnn_mask = gnn_embeddings[mask].detach().cpu().numpy()

        embeddings_to_plot["Fused embedding"] = fused_mask
        labels_to_plot["Fused embedding"] = full_labels
        embeddings_to_plot["GNN embedding"] = gnn_mask
        labels_to_plot["GNN embedding"] = full_labels

        return embeddings_to_plot, labels_to_plot

    def _custom_data_loader(self):
        return self.dataset

    def train_dataloader(self):
        return self._custom_data_loader()

    def val_dataloader(self):
        return self._custom_data_loader()

    def test_dataloader(self):
        return self._custom_data_loader()